# Event Final - Geintegreerde Event Assembly voor Phase1
## Masterproef: Spatiotemporal Prediction and Optimization of Car Parking in Mechelen

Deze notebook vervangt `04_event_assembly`, `04b_Events_new` en `04c_event_integratie` door **een enkele, lineaire en reproduceerbare pipeline**.

**Resultaat van deze notebook:**
- een geintegreerde event-registratie op eventniveau
- een finale dagelijkse eventfeature-tabel (`events_master`) voor MAD-integratie
- transparante cleaning-, deduplicatie- en validatiebeslissingen


## 1. Doel en context

### Waarom deze consolidatie?
De vorige notebooks bevatten waardevolle maar verspreide logica:
- `04`: handmatig samengestelde kern-events (Maanrock, Hanswijkprocessie, carnaval, kermissen, KVM)
- `04b`: exploratie van de UGent/Stad Mechelen eventdump
- `04c`: eerste integratie en deduplicatie

Probleem: overlap, workarounds, dubbele cleaning en niet-lineaire documentatie.

### Methodologische keuzes in deze finale versie
1. We reconstrueren de **kernlogica van 04** expliciet en deterministisch.
2. We verwerken de **UGent-bron vanaf raw CSV** met projectrelevante filters.
3. We doen deduplicatie op **eventniveau** (niet enkel daily) om informatieverlies te beperken.
4. We aggregeren pas op het einde naar daily features met de bestaande `src.phase1.events`-logica.
5. We exporteren zowel auditbare tussenoutput als finale MAD-input.

### Conflicten tussen oude versies en oplossing
- `04b` was exploratief (diagnostiek + ad-hoc cells), geen finale pipeline.
- `04c` merge-de daily output van 04 met UGent, wat minder fijnmazig is voor event-level traceability.
- Deze notebook kiest daarom voor: **integratie op eventniveau -> daarna daily aggregatie**.


## 2. Setup en inputbronnen

In [1]:
from __future__ import annotations

from pathlib import Path
from difflib import SequenceMatcher
import csv
import re
import sys
import time

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.parent.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.project_config import DEFAULT_EVENTS_END, DEFAULT_PROJECT_START, get_project_paths
from src.phase1.events import (
    aggregate_events_daily,
    build_events_master,
    date_range_list,
    extract_kvm_home_matches_wikipedia,
    make_event_records,
)

paths = get_project_paths(PROJECT_ROOT)
PROJECT_START = pd.Timestamp(DEFAULT_PROJECT_START)
PROJECT_END = pd.Timestamp(DEFAULT_EVENTS_END)

RAW_UG_PATH = paths.data_raw / 'UGent_evenementen_mechelen.csv'
KVM_HIST_CACHE = paths.data_raw / 'kvm_home_matches_hist_2019_2025.csv'

OUT_EVENTS_REGISTRY = paths.data_intermediate / 'events_registry_final.parquet'
OUT_EVENTS_DROPPED_DUP = paths.data_intermediate / 'events_registry_ugent_dropped_duplicates.parquet'
OUT_EVENTS_MASTER_FINAL = paths.data_intermediate / 'events_master_event_final.parquet'
OUT_EVENTS_MASTER_CANONICAL = paths.data_intermediate / 'events_master.parquet'  # compatibel met MAD pipeline

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'Datumbereik : {PROJECT_START.date()} -> {PROJECT_END.date()}')


PROJECT_ROOT: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2
Datumbereik : 2019-01-01 -> 2026-12-31


In [2]:
inputs = pd.DataFrame([
    {'bron': 'UGent/Stad Mechelen eventdump', 'pad': str(RAW_UG_PATH), 'bestaat': RAW_UG_PATH.exists()},
    {'bron': 'Optionele cache KVM historiek', 'pad': str(KVM_HIST_CACHE), 'bestaat': KVM_HIST_CACHE.exists()},
])
display(inputs)


,bron,pad,bestaat
0,UGent/Stad Mechelen eventdump,/Users/emilevandevoorde/Documents/mp_mechelen_...,True
1,Optionele cache KVM historiek,/Users/emilevandevoorde/Documents/mp_mechelen_...,False


## 3. Reconstructie van de 04-kernbron (curated baseline)

Deze sectie herbouwt de inhoudelijke kern uit `04_event_assembly` in compacte, gecontroleerde vorm.

### Inhoud die behouden wordt uit 04
- Maanrock
- Hanswijkprocessie
- Carnavalsfoor + Carnavalsstoet
- Zomerkermis Grote Markt + Herfstkermis Veemarkt
- KVM thuiswedstrijden (historiek + manuele 2025-2026)

### Belangrijk verschil t.o.v. oude 04
Voor KVM historiek voorzien we een **offline fallbacklijst** zodat de notebook reproduceerbaar blijft zonder internettoegang.


In [3]:
EVENT_TYPES = {
    'is_football_day': 'football',
    'is_sport_day': 'sport',
    'is_festival_day': 'festival',
    'is_procession_day': 'procession',
    'is_kermis_day': 'kermis',
    'is_markt_day': 'markt',
    'is_carnival_day': 'carnival',
    'is_other_day': 'other',
}

CONFIDENCE_RANK = {'verified': 0, 'estimated': 1, 'covid_modified': 2}


def validate_event_registry(df: pd.DataFrame, label: str) -> None:
    required = [
        'date', 'event_name', 'event_type', 'event_scale',
        'expected_attendance', 'football_kickoff_hour',
        'data_confidence', 'is_multiday_event'
    ]
    missing = [c for c in required if c not in df.columns]
    assert not missing, f"{label}: ontbrekende kolommen: {missing}"

    d = df.copy()
    d['date'] = pd.to_datetime(d['date'], errors='coerce').dt.normalize()

    print(f'=== {label} ===')
    print('shape              :', d.shape)
    print('date range         :', d['date'].min(), '->', d['date'].max())
    print('unique dates       :', d['date'].nunique())
    print('missing date       :', int(d['date'].isna().sum()))
    print('missing event_name :', int(d['event_name'].isna().sum()))
    print('event_type dist    :', d['event_type'].value_counts().to_dict())
    print('confidence dist    :', d['data_confidence'].value_counts().to_dict())

    dup_same_name = d.duplicated(subset=['date', 'event_type', 'event_name']).sum()
    print('dup(date,type,name):', int(dup_same_name))



In [4]:
# 3.1 Maanrock
maanrock_cfg = [
    ('2021-08-28', '2021-08-29', 'Maanrock (beperkte editie)', 'festival', 1, 2000, 'covid_modified', 1),
    ('2022-08-25', '2022-08-28', 'Maanrock', 'festival', 3, 80000, 'verified', 1),
    ('2023-08-24', '2023-08-27', 'Maanrock', 'festival', 3, 100000, 'verified', 1),
    ('2024-08-22', '2024-08-25', 'Maanrock', 'festival', 3, 90000, 'verified', 1),
    ('2025-08-21', '2025-08-24', 'Maanrock', 'festival', 3, 90000, 'verified', 1),
    ('2026-08-27', '2026-08-30', 'Maanrock', 'festival', 3, 90000, 'verified', 1),
]

maanrock_parts = []
for start, end, name, etype, scale, att, conf, multi in maanrock_cfg:
    maanrock_parts.append(make_event_records(
        dates=date_range_list(start, end),
        event_name=name,
        event_type=etype,
        event_scale=scale,
        expected_attendance=att,
        data_confidence=conf,
        is_multiday_event=multi,
    ))

df_maanrock = pd.concat(maanrock_parts, ignore_index=True)

# 3.2 Hanswijkprocessie
hanswijk_cfg = [
    ('2021-05-23', 'covid_modified', 5000),
    ('2022-05-29', 'verified', 50000),
    ('2023-05-21', 'verified', 50000),
    ('2024-05-12', 'verified', 50000),
    ('2025-05-25', 'verified', 50000),
    ('2026-05-10', 'verified', 50000),
]
hanswijk_parts = []
for d, conf, att in hanswijk_cfg:
    hanswijk_parts.append(make_event_records(
        dates=[d],
        event_name='Hanswijkprocessie',
        event_type='procession',
        event_scale=2,
        expected_attendance=att,
        data_confidence=conf,
        is_multiday_event=0,
    ))

df_hanswijk = pd.concat(hanswijk_parts, ignore_index=True)

# 3.3 Carnaval
carnaval_cfg = [
    ('2022-03-26', '2022-03-30', '2022-03-27', 'estimated'),
    ('2023-03-25', '2023-03-29', '2023-03-26', 'estimated'),
    ('2024-03-30', '2024-04-03', '2024-03-31', 'estimated'),
    ('2025-03-29', '2025-04-02', '2025-03-30', 'verified'),
    ('2026-03-28', '2026-04-01', '2026-03-29', 'verified'),
]

carnaval_parts = []
for foor_start, foor_end, stoet_date, conf in carnaval_cfg:
    year = pd.Timestamp(foor_start).year
    carnaval_parts.append(make_event_records(
        dates=date_range_list(foor_start, foor_end),
        event_name=f'Carnavalfoor {year}',
        event_type='carnival',
        event_scale=1,
        expected_attendance=np.nan,
        data_confidence=conf,
        is_multiday_event=1,
    ))
    carnaval_parts.append(make_event_records(
        dates=[stoet_date],
        event_name=f'Carnavalsstoet {year}',
        event_type='carnival',
        event_scale=2,
        expected_attendance=20000,
        data_confidence=conf,
        is_multiday_event=0,
    ))

df_carnaval = pd.concat(carnaval_parts, ignore_index=True)

# 3.4 Kermissen
kermis_cfg = [
    (2020, '2020-06-24', '2020-07-10', '2020-10-02', '2020-10-18', 'estimated'),
    (2021, '2021-06-23', '2021-07-09', '2021-10-01', '2021-10-17', 'estimated'),
    (2022, '2022-06-29', '2022-07-15', '2022-09-30', '2022-10-16', 'estimated'),
    (2023, '2023-06-28', '2023-07-14', '2023-10-06', '2023-10-22', 'estimated'),
    (2024, '2024-06-26', '2024-07-12', '2024-10-04', '2024-10-20', 'estimated'),
    (2025, '2025-06-25', '2025-07-11', '2025-10-03', '2025-10-19', 'verified'),
]

kermis_parts = []
for year, zs, ze, hs, he, conf in kermis_cfg:
    kermis_parts.append(make_event_records(
        dates=date_range_list(zs, ze),
        event_name=f'Zomerkermis Grote Markt {year}',
        event_type='kermis',
        event_scale=2,
        expected_attendance=np.nan,
        data_confidence=conf,
        is_multiday_event=1,
    ))
    kermis_parts.append(make_event_records(
        dates=date_range_list(hs, he),
        event_name=f'Herfstkermis Veemarkt {year}',
        event_type='kermis',
        event_scale=2,
        expected_attendance=np.nan,
        data_confidence=conf,
        is_multiday_event=1,
    ))

df_kermis = pd.concat(kermis_parts, ignore_index=True)

print('Kernblokken opgebouwd.')
print('Maanrock rows  :', len(df_maanrock))
print('Hanswijk rows  :', len(df_hanswijk))
print('Carnaval rows  :', len(df_carnaval))
print('Kermis rows    :', len(df_kermis))


Kernblokken opgebouwd.
Maanrock rows  : 22
Hanswijk rows  : 6
Carnaval rows  : 30
Kermis rows    : 204


In [5]:
# 3.5 KVM thuiswedstrijden
# Offline fallback (deterministisch snapshot). Wordt gebruikt als webextractie niet beschikbaar is.
KVM_HIST_OFFLINE = [
    [
        "2020-07-18",
        18.0
    ],
    [
        "2020-08-09",
        18.0
    ],
    [
        "2020-08-22",
        18.0
    ],
    [
        "2020-09-12",
        18.0
    ],
    [
        "2020-09-26",
        18.0
    ],
    [
        "2020-10-17",
        18.0
    ],
    [
        "2020-11-06",
        20.75
    ],
    [
        "2020-11-29",
        18.0
    ],
    [
        "2020-12-13",
        18.0
    ],
    [
        "2020-12-17",
        20.75
    ],
    [
        "2020-12-27",
        18.0
    ],
    [
        "2021-01-10",
        18.0
    ],
    [
        "2021-01-20",
        20.75
    ],
    [
        "2021-01-23",
        18.0
    ],
    [
        "2021-01-30",
        18.0
    ],
    [
        "2021-02-03",
        20.75
    ],
    [
        "2021-02-19",
        20.75
    ],
    [
        "2021-03-21",
        18.0
    ],
    [
        "2021-04-10",
        18.0
    ],
    [
        "2021-05-09",
        18.0
    ],
    [
        "2021-05-16",
        18.0
    ],
    [
        "2021-05-22",
        18.0
    ],
    [
        "2021-07-09",
        20.75
    ],
    [
        "2021-07-17",
        18.0
    ],
    [
        "2021-07-25",
        18.0
    ],
    [
        "2021-08-07",
        18.0
    ],
    [
        "2021-08-22",
        18.0
    ],
    [
        "2021-09-18",
        18.0
    ],
    [
        "2021-10-01",
        20.75
    ],
    [
        "2021-10-23",
        18.0
    ],
    [
        "2021-10-26",
        20.75
    ],
    [
        "2021-11-06",
        18.0
    ],
    [
        "2021-11-19",
        20.75
    ],
    [
        "2021-12-01",
        20.75
    ],
    [
        "2021-12-05",
        18.0
    ],
    [
        "2021-12-15",
        20.75
    ],
    [
        "2021-12-22",
        20.75
    ],
    [
        "2021-12-27",
        20.75
    ],
    [
        "2022-01-23",
        18.0
    ],
    [
        "2022-02-06",
        18.0
    ],
    [
        "2022-02-12",
        18.0
    ],
    [
        "2022-02-26",
        18.0
    ],
    [
        "2022-03-12",
        18.0
    ],
    [
        "2022-04-02",
        18.0
    ],
    [
        "2022-04-23",
        18.0
    ],
    [
        "2022-05-10",
        20.75
    ],
    [
        "2022-05-21",
        18.0
    ],
    [
        "2022-07-09",
        18.0
    ],
    [
        "2022-07-13",
        20.75
    ],
    [
        "2022-07-16",
        18.0
    ],
    [
        "2022-07-24",
        18.0
    ],
    [
        "2022-08-06",
        18.0
    ],
    [
        "2022-08-21",
        18.0
    ],
    [
        "2022-09-03",
        18.0
    ],
    [
        "2022-09-17",
        18.0
    ],
    [
        "2022-10-09",
        18.0
    ],
    [
        "2022-10-19",
        20.75
    ],
    [
        "2022-10-22",
        18.0
    ],
    [
        "2022-11-05",
        18.0
    ],
    [
        "2022-12-20",
        20.75
    ],
    [
        "2022-12-23",
        20.75
    ],
    [
        "2023-01-14",
        18.0
    ],
    [
        "2023-01-21",
        18.0
    ],
    [
        "2023-02-05",
        18.0
    ],
    [
        "2023-02-28",
        20.75
    ],
    [
        "2023-07-14",
        20.75
    ],
    [
        "2023-08-06",
        18.0
    ],
    [
        "2023-08-19",
        18.0
    ],
    [
        "2023-09-02",
        18.0
    ],
    [
        "2023-09-23",
        18.0
    ],
    [
        "2023-09-30",
        18.0
    ],
    [
        "2023-10-28",
        18.0
    ],
    [
        "2023-11-11",
        18.0
    ],
    [
        "2023-12-03",
        18.0
    ],
    [
        "2023-12-10",
        18.0
    ],
    [
        "2023-12-20",
        20.75
    ],
    [
        "2024-01-28",
        18.0
    ],
    [
        "2024-02-01",
        20.75
    ],
    [
        "2024-02-11",
        18.0
    ],
    [
        "2024-02-24",
        18.0
    ],
    [
        "2024-03-08",
        20.75
    ],
    [
        "2024-04-05",
        20.75
    ],
    [
        "2024-04-13",
        18.0
    ],
    [
        "2024-04-27",
        18.0
    ],
    [
        "2024-05-04",
        18.0
    ],
    [
        "2024-05-25",
        18.0
    ],
    [
        "2024-08-03",
        18.0
    ],
    [
        "2024-09-22",
        18.0
    ],
    [
        "2025-02-08",
        18.0
    ],
    [
        "2025-02-21",
        20.75
    ],
    [
        "2025-03-16",
        18.0
    ],
    [
        "2025-04-04",
        20.75
    ],
    [
        "2025-04-13",
        18.0
    ],
    [
        "2025-04-22",
        20.75
    ],
    [
        "2025-05-10",
        18.0
    ],
    [
        "2025-05-18",
        18.0
    ]
]

KVM_2526_MANUAL = [
    ('2025-08-01', 'Club Brugge', 18.0, 'Pro League'),
    ('2025-08-16', 'KAA Gent', 20.75, 'Pro League'),
    ('2025-08-30', 'La Louviere', 18.0, 'Pro League'),
    ('2025-09-21', 'Cercle Brugge', 18.0, 'Pro League'),
    ('2025-10-04', 'Sint-Truidense VV', 18.0, 'Pro League'),
    ('2025-10-24', 'OH Leuven', 20.75, 'Pro League'),
    ('2025-11-08', 'Union Saint-Gilloise', 18.0, 'Pro League'),
    ('2025-11-29', 'Standard Luik', 18.0, 'Pro League'),
    ('2025-12-06', 'Sporting Charleroi', 18.0, 'Pro League'),
    ('2025-12-27', 'FCV Dender', 18.0, 'Pro League'),
    ('2026-01-24', 'Westerlo', 18.0, 'Pro League'),
    ('2026-02-07', 'Royal Antwerp FC', 20.75, 'Pro League'),
    ('2026-02-14', 'KRC Genk', 18.0, 'Pro League'),
    ('2026-02-28', 'SV Zulte Waregem', 18.0, 'Pro League'),
    ('2026-03-14', 'RSC Anderlecht', 20.75, 'Pro League'),
]

ENABLE_ONLINE_KVM_REFRESH = False  # zet op True als je Wikipedia refresh expliciet wil proberen


def load_kvm_historical(cache_path: Path, enable_online_refresh: bool = False) -> pd.DataFrame:
    if cache_path.exists() and not enable_online_refresh:
        c = pd.read_csv(cache_path)
        c['date'] = pd.to_datetime(c['date']).dt.normalize()
        return c

    online_parts = []
    if enable_online_refresh:
        print('Online KVM extractie actief...')
        for season_start in [2019, 2020, 2021, 2022, 2023, 2024]:
            try:
                d = extract_kvm_home_matches_wikipedia(season_start)
                if len(d) > 0:
                    online_parts.append(d)
                time.sleep(1)
            except Exception:
                pass

    if online_parts:
        hist = (
            pd.concat(online_parts, ignore_index=True)
            .drop_duplicates(subset=['date'])
            .sort_values('date')
            .reset_index(drop=True)
        )
        hist.to_csv(cache_path, index=False)
        print(f'Online historiek opgeslagen in cache: {cache_path}')
        return hist

    # Offline fallback: volledige lokale lijst
    rows = []
    for d, kickoff in KVM_HIST_OFFLINE:
        rows.append({
            'date': pd.Timestamp(d).normalize(),
            'event_name': 'KVM thuiswedstrijd',
            'event_type': 'football',
            'event_scale': 2,
            'expected_attendance': 11000.0,
            'football_kickoff_hour': float(kickoff),
            'data_confidence': 'verified',
            'is_multiday_event': 0,
        })
    hist = pd.DataFrame(rows).sort_values('date').reset_index(drop=True)
    print('Offline KVM fallback gebruikt (geen online extractie/cache).')
    return hist


df_kvm_hist = load_kvm_historical(KVM_HIST_CACHE, ENABLE_ONLINE_KVM_REFRESH)

kvm_2526_parts = []
for d, opp, kickoff, comp in KVM_2526_MANUAL:
    kvm_2526_parts.append(make_event_records(
        dates=[d],
        event_name=f'KVM thuis vs {opp}',
        event_type='football',
        event_scale=2,
        expected_attendance=11000,
        football_kickoff_hour=kickoff,
        data_confidence='verified',
        is_multiday_event=0,
    ))

df_kvm_2526 = pd.concat(kvm_2526_parts, ignore_index=True)

df_kvm = (
    pd.concat([df_kvm_hist, df_kvm_2526], ignore_index=True)
    .drop_duplicates(subset=['date'])
    .sort_values('date')
    .reset_index(drop=True)
)

print('KVM hist rows :', len(df_kvm_hist))
print('KVM 2526 rows :', len(df_kvm_2526))
print('KVM total rows:', len(df_kvm))


Offline KVM fallback gebruikt (geen online extractie/cache).
KVM hist rows : 96
KVM 2526 rows : 15
KVM total rows: 111


In [6]:
# 3.6 Baseline event registry (uit 04-logica)
baseline_registry = pd.concat(
    [df_maanrock, df_hanswijk, df_carnaval, df_kermis, df_kvm],
    ignore_index=True,
)
baseline_registry['date'] = pd.to_datetime(baseline_registry['date'], errors='coerce').dt.normalize()

baseline_registry = baseline_registry[
    (baseline_registry['date'] >= PROJECT_START) &
    (baseline_registry['date'] <= PROJECT_END)
].copy()

baseline_registry['source'] = 'baseline_04_curated'
baseline_registry['source_priority'] = 0
baseline_registry['external_id'] = np.nan
baseline_registry['raw_type'] = np.nan
baseline_registry['raw_status'] = np.nan
baseline_registry['raw_owner'] = np.nan
baseline_registry['raw_reference'] = np.nan
baseline_registry['raw_description'] = baseline_registry['event_name']

validate_event_registry(baseline_registry, 'Baseline registry')
display(baseline_registry.head(10))


=== Baseline registry ===
shape              : (373, 16)
date range         : 2020-06-24 00:00:00 -> 2026-08-30 00:00:00
unique dates       : 360
missing date       : 0
missing event_name : 0
event_type dist    : {'kermis': 204, 'football': 111, 'carnival': 30, 'festival': 22, 'procession': 6}
confidence dist    : {'estimated': 188, 'verified': 182, 'covid_modified': 3}
dup(date,type,name): 0


,date,event_name,event_type,event_scale,expected_attendance,football_kickoff_hour,data_confidence,is_multiday_event,source,source_priority,external_id,raw_type,raw_status,raw_owner,raw_reference,raw_description
0,2021-08-28,Maanrock (beperkte editie),festival,1,2000.0,NaN,covid_modified,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Maanrock (beperkte editie)
1,2021-08-29,Maanrock (beperkte editie),festival,1,2000.0,NaN,covid_modified,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Maanrock (beperkte editie)
2,2022-08-25,Maanrock,festival,3,80000.0,NaN,verified,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Maanrock
3,2022-08-26,Maanrock,festival,3,80000.0,NaN,verified,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Maanrock
4,2022-08-27,Maanrock,festival,3,80000.0,NaN,verified,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Maanrock
5,2022-08-28,Maanrock,festival,3,80000.0,NaN,verified,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Maanrock
6,2023-08-24,Maanrock,festival,3,100000.0,NaN,verified,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Maanrock
7,2023-08-25,Maanrock,festival,3,100000.0,NaN,verified,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Maanrock
8,2023-08-26,Maanrock,festival,3,100000.0,NaN,verified,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Maanrock
9,2023-08-27,Maanrock,festival,3,100000.0,NaN,verified,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Maanrock


## 4. Datainspectie van de UGent/Stad Mechelen bron (04b-logica)

Deze bron bevat veel extra records, maar ook operationele meldingen zonder relevante parkeerimpact.
Daarom volgen inspectie en gefaseerde filtering.


In [7]:
def detect_delimiter(path: Path, default: str = ',') -> str:
    sample = path.read_text(encoding='utf-8', errors='ignore')[:20000]
    try:
        dialect = csv.Sniffer().sniff(sample, delimiters=[',', ';', '	', '|'])
        return dialect.delimiter
    except csv.Error:
        return default

ug_delim = detect_delimiter(RAW_UG_PATH)
ug_raw = pd.read_csv(RAW_UG_PATH, sep=ug_delim)

print('Delimiter            :', repr(ug_delim))
print('Raw shape            :', ug_raw.shape)
print('Kolommen             :', list(ug_raw.columns))
print('Status verdeling     :', ug_raw['Status'].value_counts(dropna=False).head(8).to_dict())
print('Type top             :', ug_raw['Type'].value_counts(dropna=False).head(10).to_dict())
print('Beheerder top        :', ug_raw['Beheerder'].value_counts(dropna=False).head(10).to_dict())
print('Volledige rijdup      :', int(ug_raw.duplicated().sum()))
print('Duplicaat op GipodId :', int(ug_raw.duplicated(subset=['GipodId']).sum()))

display(ug_raw.head(5))


Delimiter            : ','
Raw shape            : (4817, 9)
Kolommen             : ['GipodId', 'Beschrijving', 'Referentie', 'Beheerder', 'StartTijdstip', 'EindTijdstip', 'Status', 'Type', 'PointOnSurface']
Status verdeling     : {'Afgelopen': 4815, 'Lopende': 2}
Type top             : {'Andere': 2614, 'Sport': 1530, 'Feest/kermis': 331, 'Ambulante handel': 183, 'Speelstraat': 82, 'Markt': 57, 'Terras evenement': 18, 'Andere, Feest/kermis': 1, 'Feest/kermis, Andere': 1}
Beheerder top        : {'Stad Mechelen': 4371, 'Sport Vlaanderen': 217, 'Gemeente Zemst': 38, 'Gemeente Willebroek': 28, 'Gemeente Rumst': 18, 'Gemeente Boom': 11, 'Gemeente Sint-Katelijne-Waver': 10, 'Gemeente Kontich': 9, 'Gemeente Bornem': 6, 'Gemeente Heist-op-den-Berg': 6}
Volledige rijdup      : 0
Duplicaat op GipodId : 50


,GipodId,Beschrijving,Referentie,Beheerder,StartTijdstip,EindTijdstip,Status,Type,PointOnSurface
0,7259080,de 1000 km van kom op tegen kanker,BEG-1000KMKomOpTegenKanker_2020,Gemeente Begijnendijk,2020-05-22 06:00:00.0000000,2020-05-22 17:00:00.0000000,Afgelopen,Andere,POINT (230353.7922591309 174269.4566425)
1,7259083,de 1000 km van kom op tegen kanker,BEG-1000KMKomOpTegenKanker_20200523,Gemeente Begijnendijk,2020-05-23 06:00:00.0000000,2020-05-23 18:00:00.0000000,Afgelopen,Andere,POINT (164838.11271400272 174026.137065)
2,7803577,"1981 Zemst, Tervuursesteenweg 30: Leveren van ...",1159195,Gemeente Zemst,2020-06-22 04:00:00.0000000,2020-06-25 18:00:00.0000000,Afgelopen,Andere,POINT (158276.32231052575 188148.0437028835)
3,8334857,"2812 Mechelen, Toekomststraat 24: Herstellings...",1188507,Stad Mechelen,2020-09-02 05:00:00.0000000,2020-09-03 16:00:00.0000000,Afgelopen,Andere,POINT (159435.690965948 189068.1384565405)
4,8346399,"2800 Mechelen, Oude Antwerpsebaan 73: Parkeerv...",1188525,Stad Mechelen,2020-09-03 05:00:00.0000000,2020-09-03 16:00:00.0000000,Afgelopen,Andere,POINT (157026.58384379954 192348.9348210725)


## 5. Cleaning en filtering van UGent-events

### Filterstrategie (projectrelevantie)

1. **Bronfilter (uit 04b):**
   - behoud enkel `Beheerder` in `{Stad Mechelen, Sport Vlaanderen}`
   - verwijder duidelijke infrastructuur/werkmeldingen (`werf`, `nutswerken`, `laad`, ...)

2. **Technische validiteit:**
   - parsebare start/eindtijd
   - niet-negatieve duurtijd
   - geldige status (`Afgelopen`, `Lopende`)
   - deduplicatie op `GipodId` (laatste eindtijd)

3. **Relevantiefilter (aanscherping uit 04c + feedback):**
   - verwijder operationele/administratieve meldingen (`Stadsdiensten`, `Verhuis`, `Container`, ...)
   - verwijder lage-impact types zonder eventsignaal
   - `Type='Andere'` enkel behouden indien tekst duidelijk eventsignaal bevat

Elke stap rapporteert hoeveel rijen overblijven.


In [8]:
clean_steps = []

def log_step(name: str, df: pd.DataFrame):
    clean_steps.append({'stap': name, 'rows': int(len(df))})

u = ug_raw.copy()
log_step('raw_input', u)

# 5.1 Bronfilter
allowed_owners = {'Stad Mechelen', 'Sport Vlaanderen'}
blocked_terms = {'2811', '2812', 'herstellingswerken', 'nutswerken', 'werf', 'laad', 'loszone', 'werken'}

u = u[u['Beheerder'].isin(allowed_owners)].copy()
log_step('owner_filter', u)

blocked_pattern = '|'.join(sorted(re.escape(t) for t in blocked_terms))
mask_blocked_infra = u['Beschrijving'].fillna('').str.lower().str.contains(blocked_pattern, regex=True)
u = u[~mask_blocked_infra].copy()
log_step('blocked_infra_filter', u)

# 5.2 Technische validiteit
u['StartTijdstip_dt'] = pd.to_datetime(u['StartTijdstip'], errors='coerce', utc=True)
u['EindTijdstip_dt'] = pd.to_datetime(u['EindTijdstip'], errors='coerce', utc=True)
u['start_local'] = u['StartTijdstip_dt'].dt.tz_convert('Europe/Brussels')
u['end_local'] = u['EindTijdstip_dt'].dt.tz_convert('Europe/Brussels')
u['date'] = u['start_local'].dt.normalize().dt.tz_localize(None)
u['duration_hours'] = (u['end_local'] - u['start_local']).dt.total_seconds() / 3600

u = u[u['Status'].isin(['Afgelopen', 'Lopende'])].copy()
log_step('status_filter', u)

u = u[u['date'].notna() & u['duration_hours'].notna()].copy()
log_step('datetime_valid_filter', u)

u = u[u['duration_hours'] >= 0].copy()
log_step('non_negative_duration', u)

u = u.sort_values(['GipodId', 'EindTijdstip_dt']).drop_duplicates('GipodId', keep='last')
log_step('dedup_gipodid', u)

# 5.3 Relevantiefilter
text_blob = (u['Beschrijving'].fillna('') + ' ' + u['Type'].fillna('')).str.lower()
type_l = u['Type'].fillna('').str.lower()

relevant_kw = (
    r'carnaval|kermis|foor|festival|concert|muziek|processie|stoet|'
    r'sport|voetbal|wedstrijd|maanrock|hanswijk|malinwa|kv\s*mechelen|'
    r'braderie|jaarmarkt|rommelmarkt|optocht|koers|wedstrijd'
)
irrelevant_kw = (
    r'stadsdiensten?|verhuis|verhuizing|verhuislift|container|kraan|'
    r'signalisatie|huisvuil|afval|ophaling|levering|'
    r'inname\s+openbaar\s+domein|toelating|vergunning|nutswerk|werf'
)

mask_relevant_signal = text_blob.str.contains(relevant_kw, regex=True)
mask_irrelevant_signal = text_blob.str.contains(irrelevant_kw, regex=True)

low_impact_types = {'ambulante handel', 'speelstraat', 'terras evenement'}
mask_low_impact_no_signal = type_l.isin(low_impact_types) & ~mask_relevant_signal
mask_other_no_signal = (type_l == 'andere') & ~mask_relevant_signal

mask_drop_relevance = mask_irrelevant_signal | mask_low_impact_no_signal | mask_other_no_signal
u_dropped_relevance = u[mask_drop_relevance].copy()

u = u[~mask_drop_relevance].copy()
log_step('relevance_filter', u)

clean_summary = pd.DataFrame(clean_steps)
clean_summary['removed_vs_prev'] = clean_summary['rows'].shift(1) - clean_summary['rows']
clean_summary['removed_vs_prev'] = clean_summary['removed_vs_prev'].fillna(0).astype(int)
display(clean_summary)

print('Relevance removal breakdown:')
print({
    'irrelevant_signal': int(mask_irrelevant_signal.sum()),
    'low_impact_no_signal': int(mask_low_impact_no_signal.sum()),
    'other_no_signal': int(mask_other_no_signal.sum()),
})
print('Date range cleaned UGent:', u['date'].min(), '->', u['date'].max())

print('\nVoorbeelden verwijderd (relevantie):')
display(u_dropped_relevance[['GipodId', 'Type', 'Beschrijving', 'Beheerder', 'date']].head(15))


,stap,rows,removed_vs_prev
0,raw_input,4817,0
1,owner_filter,4588,229
2,blocked_infra_filter,1767,2821
3,status_filter,1767,0
4,datetime_valid_filter,1767,0
5,non_negative_duration,1767,0
6,dedup_gipodid,1717,50
7,relevance_filter,805,912


Relevance removal breakdown:
{'irrelevant_signal': 29, 'low_impact_no_signal': 203, 'other_no_signal': 697}
Date range cleaned UGent: 2021-05-01 00:00:00 -> 2025-12-12 00:00:00

Voorbeelden verwijderd (relevantie):


,GipodId,Type,Beschrijving,Beheerder,date
499,8416370,Andere,"2800 Mechelen, Korenmarkt 9: Andere",Stad Mechelen,2020-10-13
78,8427532,Andere,"2800 Mechelen, Donkerlei 76: Stadsdiensten (en...",Stad Mechelen,2020-09-14
334,8534428,Andere,"2800 Mechelen, Grote Markt: Verhuis",Stad Mechelen,2020-10-01
363,8547557,Andere,"2800 Mechelen, Zelestraat 65: Stadsdiensten (e...",Stad Mechelen,2020-10-05
468,8686789,Andere,"2800 Mechelen, Generaal de ceuninckstraat zijg...",Stad Mechelen,2020-10-12
504,8689403,Andere,"2800 Mechelen, Adegemstraat op brede gedeelte ...",Stad Mechelen,2020-10-13
521,8732464,Andere,"2800 Mechelen, Reuzenstraat 1: Stadsdiensten (...",Stad Mechelen,2020-10-15
967,8910413,Andere,"2800 Mechelen, Zennestraat 10: Stadsdiensten (...",Stad Mechelen,2020-11-10
929,8910659,Andere,"2800 Mechelen, Zelestraat 67: Stadsdiensten (e...",Stad Mechelen,2020-11-09
995,8911387,Andere,"2800 Mechelen, Groenstraat 22: Stadsdiensten (...",Stad Mechelen,2020-11-12


## 6. Harmonisatie naar eenduidig event-schema

UGent-records worden gemapt naar hetzelfde schema als de curated baseline:
- `event_type` in `{football, festival, procession, kermis, carnival, other}`
- `event_scale` op schaal 1-3
- `expected_attendance` als heuristische proxy (transparant, reproduceerbaar)

Deze mapping is een **modelleerkeuze** en blijft een benadering; daarom bewaren we de ruwe bronkolommen voor audit.


In [9]:
def normalize_raw_type(raw_type: str) -> str:
    r = str(raw_type).strip().lower()
    mapping = {
        'sport': 'sport',
        'markt': 'markt',
        'feest/kermis': 'kermis',
        'andere, feest/kermis': 'kermis',
        'feest/kermis, andere': 'kermis',
        'ambulante handel': 'markt',
        'speelstraat': 'other',
        'terras evenement': 'other',
        'andere': 'other',
    }
    return mapping.get(r, 'other')


def infer_event_type(type_raw: str, desc: str) -> str:
    # 1) Start from normalized raw_type to maximize consistency.
    base = normalize_raw_type(type_raw)

    # 2) Apply explicit semantic overrides when text clearly indicates specific major type.
    t = f"{type_raw} {desc}".lower()

    if re.search(r'kv\s*mechelen|malinwa|voetbal|thuiswedstrijd|achter de kazerne', t):
        return 'football'
    if re.search(r'carnaval', t):
        return 'carnival'
    if re.search(r'processie|stoet|optocht', t):
        return 'procession'

    # Keep market-like events as market, even with broad event wording
    if base == 'markt':
        return 'markt'

    # Keep sport as sport unless explicit football override above
    if base == 'sport':
        return 'sport'

    # Only map to festival if raw type is not already specific and text is explicit
    if base == 'other' and re.search(r'festival|concert|muziek|podium|optreden', t):
        return 'festival'

    return base


def infer_event_scale(event_type: str, desc: str, duration_h: float) -> int:
    x = str(desc).lower()

    if 'maanrock' in x:
        return 3
    if 'hanswijkprocessie' in x:
        return 2

    base = {
        'football': 2,
        'sport': 2,
        'festival': 2,
        'procession': 2,
        'kermis': 1,
        'markt': 1,
        'carnival': 2,
        'other': 1,
    }.get(event_type, 1)

    if duration_h >= 72:
        base = min(3, base + 1)
    return int(base)


def infer_attendance(event_type: str, event_scale: int) -> float:
    baseline = {
        'football': 11000.0,
        'sport': 6000.0,
        'festival': 8000.0,
        'procession': 6000.0,
        'kermis': 2500.0,
        'markt': 1800.0,
        'carnival': 5000.0,
        'other': 1200.0,
    }.get(event_type, 1200.0)
    factor = {1: 0.7, 2: 1.0, 3: 1.6}.get(int(event_scale), 1.0)
    return float(baseline * factor)


def infer_confidence(status: str) -> str:
    if status == 'Afgelopen':
        return 'verified'
    return 'estimated'


def infer_family(text: str) -> str:
    t = str(text).lower()
    if 'maanrock' in t:
        return 'maanrock'
    if 'hanswijk' in t:
        return 'hanswijk'
    if 'carnaval' in t:
        return 'carnaval'
    if 'kermis' in t or 'foor' in t:
        return 'kermis'
    if re.search(r'kv\s*mechelen|malinwa|thuiswedstrijd|achter de kazerne|voetbal', t):
        return 'kvm'
    if re.search(r'markt|braderie|rommelmarkt|jaarmarkt', t):
        return 'markt'
    return 'other'

ug_std = u.copy()
ug_std['event_name'] = ug_std['Beschrijving'].fillna('').astype(str).str.slice(0, 180).str.strip()
ug_std['raw_type_norm'] = ug_std['Type'].map(normalize_raw_type)
ug_std['event_type'] = [infer_event_type(t, d) for t, d in zip(ug_std['Type'], ug_std['Beschrijving'])]
ug_std['event_scale'] = [infer_event_scale(et, d, dh) for et, d, dh in zip(ug_std['event_type'], ug_std['Beschrijving'], ug_std['duration_hours'])]
ug_std['expected_attendance'] = [infer_attendance(et, es) for et, es in zip(ug_std['event_type'], ug_std['event_scale'])]
ug_std['football_kickoff_hour'] = np.where(
    ug_std['event_type'] == 'football',
    ug_std['start_local'].dt.hour + ug_std['start_local'].dt.minute / 60.0,
    np.nan,
)
ug_std['data_confidence'] = [infer_confidence(s) for s in ug_std['Status']]
ug_std['is_multiday_event'] = (ug_std['duration_hours'] >= 24).astype('int8')
ug_std['event_family'] = [infer_family(t) for t in ug_std['event_name']]

ug_std['source'] = 'ugent_city_dump'
ug_std['source_priority'] = 1
ug_std['external_id'] = ug_std['GipodId'].astype(str)
ug_std['raw_type'] = ug_std['Type']
ug_std['raw_status'] = ug_std['Status']
ug_std['raw_owner'] = ug_std['Beheerder']
ug_std['raw_reference'] = ug_std['Referentie']
ug_std['raw_description'] = ug_std['Beschrijving']

# Consistency audit between normalized raw_type and event_type
ug_std['raw_eventtype_mismatch'] = ug_std['raw_type_norm'] != ug_std['event_type']

ug_keep_cols = [
    'date', 'event_name', 'event_type', 'event_scale', 'expected_attendance',
    'football_kickoff_hour', 'data_confidence', 'is_multiday_event',
    'source', 'source_priority', 'external_id', 'event_family',
    'raw_type', 'raw_status', 'raw_owner', 'raw_reference', 'raw_description',
    'raw_type_norm', 'raw_eventtype_mismatch'
]

ug_registry = ug_std[ug_keep_cols].copy()
validate_event_registry(ug_registry, 'UGent gestandaardiseerde registry')

print('UGent event_type verdeling:')
print(ug_registry['event_type'].value_counts().to_string())
print('\nUGent raw_type_norm verdeling:')
print(ug_registry['raw_type_norm'].value_counts().to_string())
print('\nMismatch raw_type_norm vs event_type:', int(ug_registry['raw_eventtype_mismatch'].sum()))
print('Mismatch voorbeelden:')
display(ug_registry.loc[ug_registry['raw_eventtype_mismatch'], ['date','raw_type','raw_type_norm','event_type','event_name']].head(20))

display(ug_registry.head(10))



=== UGent gestandaardiseerde registry ===
shape              : (805, 19)
date range         : 2021-05-01 00:00:00 -> 2025-12-12 00:00:00
unique dates       : 431
missing date       : 0
missing event_name : 0
event_type dist    : {'sport': 293, 'kermis': 287, 'markt': 80, 'other': 77, 'festival': 43, 'procession': 10, 'football': 9, 'carnival': 6}
confidence dist    : {'verified': 805}
dup(date,type,name): 23
UGent event_type verdeling:
event_type
sport         293
kermis        287
markt          80
other          77
festival       43
procession     10
football        9
carnival        6

UGent raw_type_norm verdeling:
raw_type_norm
sport     296
kermis    288
other     137
markt      84

Mismatch raw_type_norm vs event_type: 68
Mismatch voorbeelden:


,date,raw_type,raw_type_norm,event_type,event_name
2841,2021-10-02,Sport,sport,football,"2800 Mechelen, Oscar Van Kesbeeckstraat 43 : c..."
2842,2021-10-03,Andere,other,football,"2800 Mechelen, Kleine Nieuwedijkstraat 53 : Fa..."
2865,2021-11-07,Andere,other,procession,"2800 Mechelen, Hombekerkouter 14 : Sinte-Mette..."
2917,2022-04-10,Feest/kermis,kermis,football,"2800 Mechelen, Voetbalstraat 50 : Straatfeest ..."
2973,2022-05-13,Andere,other,festival,"2800 Mechelen, Grote Markt : preventie campagn..."
2985,2022-05-18,Andere,other,festival,"2800 Mechelen, Raghenoplein : Streetfood festi..."
3395,2023-05-14,Andere,other,procession,"2800 Mechelen, Hanswijkstraat : Hanswijkprocessie"
3049,2022-06-23,Andere,other,festival,"2800 Mechelen, Antwerpsesteenweg 92 : HAP Food..."
3221,2022-10-08,Andere,other,festival,"2800 Mechelen, Grote Markt 1 : Bierfestival Me..."
3141,2022-08-31,Andere,other,festival,"2800 Mechelen, Cortenbachplein 8 : (Pers)brief..."


,date,event_name,event_type,event_scale,expected_attendance,football_kickoff_hour,data_confidence,is_multiday_event,source,source_priority,external_id,event_family,raw_type,raw_status,raw_owner,raw_reference,raw_description,raw_type_norm,raw_eventtype_mismatch
2680,2021-05-01,"2800 Mechelen, Kruidtuin: 1 mei manifestatie (...",kermis,1,1750.0,NaN,verified,0,ugent_city_dump,1,11298029,other,Feest/kermis,Afgelopen,Stad Mechelen,1256727,"2800 Mechelen, Kruidtuin: 1 mei manifestatie (...",kermis,False
2700,2021-06-12,"2800 Mechelen, Schorsmolenstraat : Krankenmollen",kermis,1,1750.0,NaN,verified,0,ugent_city_dump,1,11298147,other,Feest/kermis,Afgelopen,Stad Mechelen,1266707,"2800 Mechelen, Schorsmolenstraat : Krankenmollen",kermis,False
2704,2021-06-15,"2800 Mechelen, Grote Markt 21 : allereerste ma...",kermis,1,1750.0,NaN,verified,1,ugent_city_dump,1,11298158,markt,Feest/kermis,Afgelopen,Stad Mechelen,1259627,"2800 Mechelen, Grote Markt 21 : allereerste ma...",kermis,False
2752,2021-08-14,"2801 Mechelen, Groenenbeemd 36 : Groenenbeemd ...",kermis,1,1750.0,NaN,verified,1,ugent_city_dump,1,11298230,other,Feest/kermis,Afgelopen,Stad Mechelen,1266110,"2801 Mechelen, Groenenbeemd 36 : Groenenbeemd ...",kermis,False
2777,2021-08-28,"2800 Mechelen, Rateaulaan 8 : burenfeest",kermis,1,1750.0,NaN,verified,1,ugent_city_dump,1,11298252,other,Feest/kermis,Afgelopen,Stad Mechelen,1259828,"2800 Mechelen, Rateaulaan 8 : burenfeest",kermis,False
2790,2021-09-03,"2800 Mechelen, Bruul : Herfstbraderie",markt,1,1260.0,NaN,verified,1,ugent_city_dump,1,11298289,markt,Markt,Afgelopen,Stad Mechelen,1277318,"2800 Mechelen, Bruul : Herfstbraderie",markt,False
2793,2021-09-04,"2801 Mechelen, Heffen-Dorp 1: Rommelmarkt Heffen",markt,1,1260.0,NaN,verified,1,ugent_city_dump,1,11298297,markt,Markt,Afgelopen,Stad Mechelen,1262518,"2801 Mechelen, Heffen-Dorp 1: Rommelmarkt Heffen",markt,False
2800,2021-09-10,"2800 Mechelen, Heembeemd: Rommelmarkt der begi...",kermis,1,1750.0,NaN,verified,1,ugent_city_dump,1,11298454,markt,Feest/kermis,Afgelopen,Stad Mechelen,1264970,"2800 Mechelen, Heembeemd: Rommelmarkt der begi...",kermis,False
2820,2021-09-23,"2800 Mechelen, Bruul 129 : Maatjesfeest 2021",kermis,2,2500.0,NaN,verified,1,ugent_city_dump,1,11298460,other,Feest/kermis,Afgelopen,Stad Mechelen,1259522,"2800 Mechelen, Bruul 129 : Maatjesfeest 2021",kermis,False
2824,2021-09-24,"2800 Mechelen, Tichelrij 15: DECAMERONE AAN DE...",kermis,2,2500.0,NaN,verified,1,ugent_city_dump,1,11298463,other,Feest/kermis,Afgelopen,Stad Mechelen,1266711,"2800 Mechelen, Tichelrij 15: DECAMERONE AAN DE...",kermis,False


## 7. Harmonisatie en integratie van bronnen (event-level)

We combineren nu de curated baseline met UGent-events.

### Deduplicatieprincipe
UGent-rijen worden als duplicaat van baseline beschouwd als ze op dezelfde datum vallen en:
1. dezelfde eventfamilie hebben (`maanrock`, `hanswijk`, `carnaval`, `kermis`, `kvm`), of
2. een sterke naamoverlap hebben (exact/substring), of
3. een hoge fuzzy gelijkenis hebben.

Zo behouden we aanvullende events, maar vermijden we dubbel tellen van reeds gekende kern-events.


In [10]:
def normalize_text(s: str) -> str:
    s = str(s).lower()
    s = re.sub(r'[^a-z0-9\s]+', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


def token_set(s: str) -> set[str]:
    stop = {
        'de', 'het', 'een', 'van', 'en', 'te', 'op', 'in', 'voor', 'met', 'door',
        'aan', 'straat', 'plein', 'mechelen', '2800', '2801', 'stad', 'markt'
    }
    return {t for t in normalize_text(s).split() if len(t) >= 3 and t not in stop}


def overlap_coeff(a: set[str], b: set[str]) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / min(len(a), len(b))


baseline_cmp = baseline_registry[['date', 'event_name', 'event_type']].copy()
baseline_cmp['name_norm'] = baseline_cmp['event_name'].map(normalize_text)
baseline_cmp['name_tokens'] = baseline_cmp['event_name'].map(token_set)
baseline_cmp['event_family'] = baseline_cmp['event_name'].map(infer_family)

ug_cmp = ug_registry.copy()
ug_cmp['name_norm'] = ug_cmp['event_name'].map(normalize_text)
ug_cmp['name_tokens'] = ug_cmp['event_name'].map(token_set)


def baseline_match_info(date_val, ug_name_norm, ug_tokens, ug_family):
    cands = baseline_cmp[baseline_cmp['date'] == date_val]
    if len(cands) == 0:
        return '', 0.0, 0.0, False, False

    best_name = ''
    best_ratio = 0.0
    best_overlap = 0.0
    has_exact = False
    has_family_match = False

    for _, row in cands.iterrows():
        bn = row['name_norm']
        bt = row['name_tokens']
        fam = row['event_family']

        if ug_name_norm and (ug_name_norm in bn or bn in ug_name_norm):
            has_exact = True

        if ug_family != 'other' and fam == ug_family:
            has_family_match = True

        ratio = SequenceMatcher(None, ug_name_norm, bn).ratio()
        ov = overlap_coeff(ug_tokens, bt)

        if (ratio > best_ratio) or (ratio == best_ratio and ov > best_overlap):
            best_ratio = ratio
            best_overlap = ov
            best_name = row['event_name']

    return best_name, best_ratio, best_overlap, has_exact, has_family_match


matches = [
    baseline_match_info(d, n, t, f)
    for d, n, t, f in zip(ug_cmp['date'], ug_cmp['name_norm'], ug_cmp['name_tokens'], ug_cmp['event_family'])
]
ug_cmp[['baseline_match_name', 'name_ratio', 'token_overlap', 'exact_name_match', 'family_match']] = pd.DataFrame(matches, index=ug_cmp.index)

ug_cmp['is_probable_duplicate'] = (
    ug_cmp['family_match'] |
    ug_cmp['exact_name_match'] |
    ((ug_cmp['name_ratio'] >= 0.90) & (ug_cmp['token_overlap'] >= 0.60))
)

ug_dropped_dup = ug_cmp[ug_cmp['is_probable_duplicate']].copy()
ug_kept = ug_cmp[~ug_cmp['is_probable_duplicate']].copy()

print('UGent rows total          :', len(ug_cmp))
print('UGent dropped duplicates  :', len(ug_dropped_dup))
print('UGent kept for integration:', len(ug_kept))

print('\nVoorbeelden gedropte duplicaten:')
display(ug_dropped_dup[['date', 'event_name', 'baseline_match_name', 'event_family', 'name_ratio', 'token_overlap']].head(20))


UGent rows total          : 805
UGent dropped duplicates  : 12
UGent kept for integration: 793

Voorbeelden gedropte duplicaten:


,date,event_name,baseline_match_name,event_family,name_ratio,token_overlap
2714,2021-06-26,"2800 Mechelen, Grote Markt : Zomerkermis",Zomerkermis Grote Markt 2021,kermis,0.492308,1.000000
2781,2021-08-29,"2800 Mechelen, Zandpoortvest 60: Maanrock @ Te...",Maanrock (beperkte editie),maanrock,0.329114,0.333333
3131,2022-08-26,"2800 Mechelen, Huidevettersstraat : Maanrock -...",Maanrock,maanrock,0.262295,1.000000
3073,2022-07-06,"2800 Mechelen, Grote Markt 21 : Animatie zomer...",Zomerkermis Grote Markt 2022,kermis,0.441558,0.666667
3082,2022-07-11,"2800 Mechelen, Grote Markt 21 : Animatie zomer...",Zomerkermis Grote Markt 2022,kermis,0.441558,0.666667
3129,2022-08-26,"2800 Mechelen, Moreelstraat 3 : Maanrock 2022",Maanrock,maanrock,0.320000,1.000000
3130,2022-08-26,"2800 Mechelen, Ganzendries : Maanrock-parade",Maanrock,maanrock,0.326531,1.000000
3323,2023-03-26,"2800 Mechelen, Lange Heergracht : Carnavalstoe...",Carnavalsstoet 2023,carnaval,0.341463,0.000000
3789,2024-03-31,"2800 Mechelen, Grote Markt : Carnavalstoet doo...",Carnavalsstoet 2024,carnaval,0.325581,0.000000
3562,2023-08-24,"2800 Mechelen, Huidevettersstraat 7 : Maanrock...",Maanrock,maanrock,0.207792,1.000000


In [11]:
# Finale event-level registry
registry_cols = [
    'date', 'event_name', 'event_type', 'event_scale', 'expected_attendance',
    'football_kickoff_hour', 'data_confidence', 'is_multiday_event',
    'source', 'source_priority', 'external_id', 'raw_type', 'raw_status',
    'raw_owner', 'raw_reference', 'raw_description'
]

baseline_final = baseline_registry[registry_cols].copy()
ugent_final = ug_kept[registry_cols].copy()

events_registry_final = pd.concat([baseline_final, ugent_final], ignore_index=True)

events_registry_final['date'] = pd.to_datetime(events_registry_final['date'], errors='coerce').dt.normalize()
events_registry_final = events_registry_final.sort_values(['date', 'source_priority', 'event_name']).reset_index(drop=True)

# defensieve dedup op exact dezelfde sleutel
events_registry_final = events_registry_final.drop_duplicates(
    subset=['date', 'event_type', 'event_name', 'source'],
    keep='first',
).reset_index(drop=True)

validate_event_registry(events_registry_final, 'Geintegreerde event registry (final)')

print('Source verdeling:')
print(events_registry_final['source'].value_counts().to_string())

display(events_registry_final.sample(min(12, len(events_registry_final)), random_state=42))


=== Geintegreerde event registry (final) ===
shape              : (1143, 16)
date range         : 2020-06-24 00:00:00 -> 2026-08-30 00:00:00
unique dates       : 693
missing date       : 0
missing event_name : 0
event_type dist    : {'kermis': 486, 'sport': 285, 'football': 120, 'markt': 80, 'other': 63, 'festival': 61, 'carnival': 33, 'procession': 15}
confidence dist    : {'verified': 952, 'estimated': 188, 'covid_modified': 3}
dup(date,type,name): 0
Source verdeling:
source
ugent_city_dump        770
baseline_04_curated    373


,date,event_name,event_type,event_scale,expected_attendance,football_kickoff_hour,data_confidence,is_multiday_event,source,source_priority,external_id,raw_type,raw_status,raw_owner,raw_reference,raw_description
158,2021-09-26,"2800 Mechelen, Stationsstraat : Garage Sale ""G...",markt,1,1260.0,NaN,verified,0,ugent_city_dump,1,11630305,Markt,Afgelopen,Stad Mechelen,1361917,"2800 Mechelen, Stationsstraat : Garage Sale ""G..."
1081,2025-09-21,Veldtoertocht Parel van Brabant TT Gravel 75km...,sport,2,6000.0,NaN,verified,0,ugent_city_dump,1,18346305,Sport,Afgelopen,Sport Vlaanderen,d73d42c5-4e9e-4b43-9b69-2257d9e01a5f,Veldtoertocht Parel van Brabant TT Gravel 75km...
291,2022-06-30,"2800 Mechelen, Leliestraat 32 : Bardok Zomert",kermis,2,2500.0,NaN,verified,1,ugent_city_dump,1,12713117,Feest/kermis,Afgelopen,Stad Mechelen,1536287,"2800 Mechelen, Leliestraat 32 : Bardok Zomert"
538,2023-07-06,Zomerkermis Grote Markt 2023,kermis,2,NaN,NaN,estimated,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Zomerkermis Grote Markt 2023
367,2022-09-03,"2800 Mechelen, Hoviusstraat 5 : Straatfeest Ho...",kermis,1,1750.0,NaN,verified,0,ugent_city_dump,1,13041202,Feest/kermis,Afgelopen,Stad Mechelen,1601232,"2800 Mechelen, Hoviusstraat 5 : Straatfeest Ho..."
793,2024-06-14,"Eetwagen, plaatsing scherm EK voetbal",football,3,17600.0,7.0,verified,1,ugent_city_dump,1,16487094,Ambulante handel,Afgelopen,Stad Mechelen,SptbMechelen20934,"Eetwagen, plaatsing scherm EK voetbal"
128,2021-08-29,Maanrock (beperkte editie),festival,1,2000.0,NaN,covid_modified,1,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,Maanrock (beperkte editie)
56,2021-05-09,"2800 Mechelen, Hanswijkstraat 67: Gebedswake i...",other,1,840.0,NaN,verified,0,ugent_city_dump,1,11402202,Andere,Afgelopen,Stad Mechelen,1307464,"2800 Mechelen, Hanswijkstraat 67: Gebedswake i..."
448,2023-01-21,"2800 Mechelen, Wilgenstraat 89 : Winterburendag",kermis,1,1750.0,NaN,verified,0,ugent_city_dump,1,13761306,Feest/kermis,Afgelopen,Stad Mechelen,1719887,"2800 Mechelen, Wilgenstraat 89 : Winterburendag"
422,2022-10-22,KVM thuiswedstrijd,football,2,11000.0,18.0,verified,0,baseline_04_curated,0,NaN,NaN,NaN,NaN,NaN,KVM thuiswedstrijd


## 8. Event engineering naar finale daily dataset (`events_master`)

We aggregeren nu event-level records naar daily features met dezelfde functie als in de codebase (`aggregate_events_daily`),
zodat downstream compatibiliteit met MAD gegarandeerd blijft.


In [12]:
events_for_agg = events_registry_final[[
    'date', 'event_name', 'event_type', 'event_scale',
    'expected_attendance', 'football_kickoff_hour',
    'data_confidence', 'is_multiday_event'
]].copy()

events_agg_final = aggregate_events_daily(events_for_agg, EVENT_TYPES)

events_master_final = build_events_master(
    events_agg=events_agg_final,
    event_flag_columns=list(EVENT_TYPES.keys()),
    project_start=str(PROJECT_START.date()),
    project_end=str(PROJECT_END.date()),
)

print('events_agg_final shape   :', events_agg_final.shape)
print('events_master_final shape:', events_master_final.shape)
print('is_event_day sum         :', int(events_master_final['is_event_day'].sum()))
print('Type-flag tellingen:')
for col in EVENT_TYPES.keys():
    print(f"  {col:<20}: {int(events_master_final[col].sum())}")

display(events_master_final.head(99))



events_agg_final shape   : (693, 15)
events_master_final shape: (2922, 15)
is_event_day sum         : 693
Type-flag tellingen:
  is_football_day     : 120
  is_sport_day        : 111
  is_festival_day     : 58
  is_procession_day   : 15
  is_kermis_day       : 390
  is_markt_day        : 69
  is_carnival_day     : 28
  is_other_day        : 63


,date,event_scale_max,n_concurrent_events,football_kickoff_hour,data_confidence,event_names_combined,is_football_day,is_sport_day,is_festival_day,is_procession_day,is_kermis_day,is_markt_day,is_carnival_day,is_other_day,is_event_day
0,2019-01-01,0,0,NaN,verified,,0,0,0,0,0,0,0,0,0
1,2019-01-02,0,0,NaN,verified,,0,0,0,0,0,0,0,0,0
2,2019-01-03,0,0,NaN,verified,,0,0,0,0,0,0,0,0,0
3,2019-01-04,0,0,NaN,verified,,0,0,0,0,0,0,0,0,0
4,2019-01-05,0,0,NaN,verified,,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,2019-04-05,0,0,NaN,verified,,0,0,0,0,0,0,0,0,0
95,2019-04-06,0,0,NaN,verified,,0,0,0,0,0,0,0,0,0
96,2019-04-07,0,0,NaN,verified,,0,0,0,0,0,0,0,0,0
97,2019-04-08,0,0,NaN,verified,,0,0,0,0,0,0,0,0,0


## 9. Validatiechecks

Hier controleren we of de finale dataset coherent, volledig en bruikbaar is voor de MAD-fase.


In [13]:
# 9.1 Schema- en integriteitschecks
expected_master_cols = {
    'date', 'event_scale_max', 'n_concurrent_events', 'football_kickoff_hour',
    'data_confidence', 'event_names_combined', 'is_event_day',
    *EVENT_TYPES.keys(),
}

assert set(events_master_final.columns) == expected_master_cols, 'events_master kolommen wijken af'
assert events_master_final['date'].is_unique, 'date moet uniek zijn in events_master'
assert events_master_final['date'].isna().sum() == 0, 'date bevat NaN'

for c in ['event_scale_max', 'n_concurrent_events', 'is_event_day', *EVENT_TYPES.keys()]:
    assert events_master_final[c].isna().sum() == 0, f'{c} bevat NaN'

flag_cols = ['is_event_day', *EVENT_TYPES.keys()]
for c in flag_cols:
    assert set(events_master_final[c].unique()).issubset({0, 1}), f'{c} bevat waarden buiten 0/1'

# date continuity
expected_days = pd.date_range(PROJECT_START, PROJECT_END, freq='D')
assert len(events_master_final) == len(expected_days), 'onverwacht aantal dagen in events_master'
assert events_master_final['date'].min() == expected_days.min(), 'startdatum mismatch'
assert events_master_final['date'].max() == expected_days.max(), 'einddatum mismatch'

print('Alle kernvalidaties geslaagd.')



Alle kernvalidaties geslaagd.


In [14]:
# 9.2 Compacte kwaliteitsrapportage
quality_report = {
    'rows_events_registry_final': int(len(events_registry_final)),
    'rows_events_master_final': int(len(events_master_final)),
    'unique_event_days': int(events_registry_final['date'].nunique()),
    'event_days_master': int(events_master_final['is_event_day'].sum()),
    'missing_event_names': int(events_registry_final['event_name'].isna().sum()),
    'dup_date_type_name': int(events_registry_final.duplicated(['date', 'event_type', 'event_name']).sum()),
    'date_min_registry': events_registry_final['date'].min(),
    'date_max_registry': events_registry_final['date'].max(),
}
quality_report


{'rows_events_registry_final': 1143,
 'rows_events_master_final': 2922,
 'unique_event_days': 693,
 'event_days_master': 693,
 'missing_event_names': 0,
 'dup_date_type_name': 0,
 'date_min_registry': Timestamp('2020-06-24 00:00:00'),
 'date_max_registry': Timestamp('2026-08-30 00:00:00')}

## 10. Export en finale output

We schrijven zowel audit-output als de finale MAD-input weg.

- `events_registry_final.parquet`: event-level bron voor traceerbaarheid
- `events_registry_ugent_dropped_duplicates.parquet`: transparantie over verwijderde UGent-duplicaten
- `events_master_event_final.parquet`: versie met expliciete final-tag
- `events_master.parquet`: **canonieke input voor volgende MAD-stap**


In [15]:
events_registry_final.to_parquet(OUT_EVENTS_REGISTRY, index=False)
ug_dropped_dup.to_parquet(OUT_EVENTS_DROPPED_DUP, index=False)
events_master_final.to_parquet(OUT_EVENTS_MASTER_FINAL, index=False)

# Canonieke naam voor downstream compatibiliteit (phase1/05_mad_assembly en src.phase1.assembly)
events_master_final.to_parquet(OUT_EVENTS_MASTER_CANONICAL, index=False)

exports = pd.DataFrame([
    {'bestand': str(OUT_EVENTS_REGISTRY), 'bestaat': OUT_EVENTS_REGISTRY.exists(), 'size_kb': round(OUT_EVENTS_REGISTRY.stat().st_size / 1024, 1)},
    {'bestand': str(OUT_EVENTS_DROPPED_DUP), 'bestaat': OUT_EVENTS_DROPPED_DUP.exists(), 'size_kb': round(OUT_EVENTS_DROPPED_DUP.stat().st_size / 1024, 1)},
    {'bestand': str(OUT_EVENTS_MASTER_FINAL), 'bestaat': OUT_EVENTS_MASTER_FINAL.exists(), 'size_kb': round(OUT_EVENTS_MASTER_FINAL.stat().st_size / 1024, 1)},
    {'bestand': str(OUT_EVENTS_MASTER_CANONICAL), 'bestaat': OUT_EVENTS_MASTER_CANONICAL.exists(), 'size_kb': round(OUT_EVENTS_MASTER_CANONICAL.stat().st_size / 1024, 1)},
])
display(exports)


,bestand,bestaat,size_kb
0,/Users/emilevandevoorde/Documents/mp_mechelen_...,True,68.2
1,/Users/emilevandevoorde/Documents/mp_mechelen_...,True,18.1
2,/Users/emilevandevoorde/Documents/mp_mechelen_...,True,54.0
3,/Users/emilevandevoorde/Documents/mp_mechelen_...,True,54.0


## 11. Samenvatting en next steps

### Samenvatting
- De logica van `04`, `04b` en `04c` is samengebracht in een eenduidige pipeline.
- Cleaning- en relevantiebeslissingen zijn expliciet en kwantitatief gerapporteerd.
- Integratie gebeurt op eventniveau met deduplicatie vooraleer daily features worden opgebouwd.
- Finale output is direct compatibel met de MAD-fase via `data_intermediate/events_master.parquet`.

### Hoe deze output verder gebruikt wordt in de volgende fase
In `phase1/05_mad_assembly.ipynb` wordt `events_master.parquet` gemerged op datum (`date_only`) met de MAD-gegevens.
Daardoor kunnen eventindicatoren (`is_event_day`, `is_kermis_day`, ...) en eventintensiteit (`event_scale_max`, `n_concurrent_events`) meteen gebruikt worden in EDA, feature engineering en modellering.

### Bekende beperkingen
- UGent eventclassificatie blijft deels heuristisch (regex + typevelden).
- Zonder externe geocodering is ruimtelijke impact per parkingzone niet volledig afleidbaar.
- KVM-historiek gebruikt standaard offline snapshot; optionele online refresh kan in de toekomst geactiveerd worden.


In [16]:
# Overzicht finale output: events_master.parquet (intuitief interpreteerbaar)
from pathlib import Path
import pandas as pd

events_master_path = PROJECT_ROOT / 'data_intermediate' / 'events_master.parquet'
em = pd.read_parquet(events_master_path).copy()
em['date'] = pd.to_datetime(em['date'], errors='coerce').dt.normalize()

flag_cols = [c for c in em.columns if c.startswith('is_') and c.endswith('_day')]
flag_cols = sorted(flag_cols, key=lambda c: (0 if c == 'is_event_day' else 1, c))

print('=== FINALE EVENTS_MASTER OVERZICHT ===')
print(f'Pad: {events_master_path}')
print(f'Shape: {em.shape[0]} rijen x {em.shape[1]} kolommen')
print(f"Datumbereik: {em['date'].min().date()} -> {em['date'].max().date()}")
print(f"Unieke datums: {em['date'].nunique()}")
print('')

print('1) Kolommen en dtypes')
display(pd.DataFrame({'kolom': em.columns, 'dtype': em.dtypes.astype(str)}))

print('2) Eventdekking op dag-niveau')
summary = {
    'eventdagen_totaal': int(em['is_event_day'].sum()),
    'niet_eventdagen': int((em['is_event_day'] == 0).sum()),
    'aandeel_eventdagen_pct': round(float(em['is_event_day'].mean() * 100), 2),
    'max_event_scale': int(em['event_scale_max'].max()),
    'max_n_concurrent_events': int(em['n_concurrent_events'].max()),
}
display(pd.Series(summary, name='waarde').to_frame())

print('3) Frequentie per eventflag')
flag_counts = pd.DataFrame({
    'event_flag': flag_cols,
    'n_dagen': [int(em[c].sum()) for c in flag_cols],
    'pct_van_alle_dagen': [round(float(em[c].mean() * 100), 2) for c in flag_cols],
}).sort_values('n_dagen', ascending=False)
display(flag_counts)

print('4) Verdeling eventintensiteit')
scale_dist = em['event_scale_max'].value_counts(dropna=False).sort_index().rename('n_dagen').to_frame()
concurrent_dist = em['n_concurrent_events'].value_counts(dropna=False).sort_index().rename('n_dagen').to_frame()
print('event_scale_max:')
display(scale_dist)
print('n_concurrent_events:')
display(concurrent_dist)

print('5) Data confidence')
conf = em['data_confidence'].value_counts(dropna=False).rename('n_dagen').to_frame()
conf['pct'] = (conf['n_dagen'] / len(em) * 100).round(2)
display(conf)

print('6) Top eventdagen op complexiteit (meerdere events op 1 dag)')
top_complex = em.loc[em['n_concurrent_events'] >= 2, [
    'date', 'n_concurrent_events', 'event_scale_max', 'event_names_combined', 'data_confidence'
]].sort_values(['n_concurrent_events', 'event_scale_max', 'date'], ascending=[False, False, True]).head(20)
display(top_complex)

print('7) Voorbeeld van "eventdag" vs "niet-eventdag"')
example_event = em.loc[em['is_event_day'] == 1, [
    'date', 'event_names_combined', 'event_scale_max', 'n_concurrent_events', 'data_confidence'
]].head(5)
example_non_event = em.loc[em['is_event_day'] == 0, [
    'date', 'event_names_combined', 'event_scale_max', 'n_concurrent_events', 'data_confidence'
]].head(5)
print('Voorbeeld eventdagen:')
display(example_event)
print('Voorbeeld niet-eventdagen:')
display(example_non_event)

print('8) Korte kwaliteitscheck voor MAD-compatibiliteit')
quality_checks = {
    'date_uniek': bool(em['date'].is_unique),
    'date_zonder_nan': int(em['date'].isna().sum()) == 0,
    'alle_flags_binary': bool(all(set(em[c].dropna().unique()).issubset({0, 1}) for c in flag_cols)),
    'geen_nan_in_core_int_cols': int(em[['event_scale_max', 'n_concurrent_events']].isna().sum().sum()) == 0,
}
display(pd.Series(quality_checks, name='ok').to_frame())

print('Conclusie: events_master.parquet is een dagelijkse, modelklare eventfeature-tabel voor MAD-assembly.')



=== FINALE EVENTS_MASTER OVERZICHT ===
Pad: /Users/emilevandevoorde/Documents/mp_mechelen_parking_v2/data_intermediate/events_master.parquet
Shape: 2922 rijen x 15 kolommen
Datumbereik: 2019-01-01 -> 2026-12-31
Unieke datums: 2922

1) Kolommen en dtypes


,kolom,dtype
date,date,datetime64[us]
event_scale_max,event_scale_max,int8
n_concurrent_events,n_concurrent_events,int8
football_kickoff_hour,football_kickoff_hour,float64
data_confidence,data_confidence,str
event_names_combined,event_names_combined,str
is_football_day,is_football_day,int8
is_sport_day,is_sport_day,int8
is_festival_day,is_festival_day,int8
is_procession_day,is_procession_day,int8


2) Eventdekking op dag-niveau


,waarde
eventdagen_totaal,693.00
niet_eventdagen,2229.00
aandeel_eventdagen_pct,23.72
max_event_scale,3.00
max_n_concurrent_events,3.00


3) Frequentie per eventflag


...

Voorbeeld niet-eventdagen:


,date,event_names_combined,event_scale_max,n_concurrent_events,data_confidence
0,2019-01-01,,0,0,verified
1,2019-01-02,,0,0,verified
2,2019-01-03,,0,0,verified
3,2019-01-04,,0,0,verified
4,2019-01-05,,0,0,verified


8) Korte kwaliteitscheck voor MAD-compatibiliteit


,ok
date_uniek,True
date_zonder_nan,True
alle_flags_binary,True
geen_nan_in_core_int_cols,True


Conclusie: events_master.parquet is een dagelijkse, modelklare eventfeature-tabel voor MAD-assembly.


## 12. Uitgebreide Methodologische Verantwoording (Thesis-stijl)

### 12.1 Doel van deze notebook binnen de thesis
Deze notebook operationaliseert de volledige event assembly voor phase1 in een enkele, reproduceerbare workflow. De primaire doelstelling is het bouwen van een betrouwbare eventfeature-laag die later zonder handmatige tussenstappen kan worden gemerged met MAD-data voor descriptieve analyse en modellering.

Concreet vervangt deze notebook de vroegere, verspreide notebooks `04_event_assembly`, `04b_Events_new` en `04c_event_integratie` door een uniforme pipeline met:
- eenduidige broninname,
- expliciete cleaningregels,
- event-level harmonisatie,
- deduplicatie,
- dagelijkse aggregatie naar `events_master.parquet`,
- en formele validatiechecks.

### 12.2 Databronnen en datakeuze
De pipeline combineert twee complementaire bronlagen:

1. **Curated baseline events (inhoudelijke kern uit vroegere 04):**
- Maanrock
- Hanswijkprocessie
- Carnavalsfoor + Carnavalsstoet
- Zomerkermis Grote Markt + Herfstkermis Veemarkt
- KVM-thuiswedstrijden

Deze laag bewaart domeinkennis en historisch gecureerde context voor centrumrelevante events.

2. **UGent/Stad Mechelen eventdump (`data_raw/UGent_evenementen_mechelen.csv`):**
Deze bron verhoogt de empirische dekking van aanvullende events, maar bevat ook operationele en administratieve records die niet noodzakelijk relevant zijn voor parkeerdrukmodellering.

De combinatie van curated en administratieve bron is methodologisch verantwoord omdat ze tegelijk:
- **inhoudelijke validiteit** bewaart (gevalideerde kern-events),
- **observatiedekking** verhoogt (meer real-world events),
- en **traceerbaarheid** mogelijk maakt (bronvelden blijven bewaard in de event registry).

### 12.3 Datareductie en cleaning: volledige stap-voor-stap verantwoording
In de huidige run werd op de UGent-bron de volgende filtering toegepast:

- `raw_input`: **4817** rijen
- `owner_filter` (enkel `Stad Mechelen` en `Sport Vlaanderen`): **4588** rijen
- `blocked_infra_filter` (infrastructuur/werken-termen): **1767** rijen
- `status_filter` (`Afgelopen`/`Lopende`): **1767** rijen
- `datetime_valid_filter` (parsebare tijd): **1767** rijen
- `non_negative_duration`: **1767** rijen
- `dedup_gipodid`: **1717** rijen
- `relevance_filter`: **805** rijen

Relevantieverwijdering gebeurde op drie expliciete criteria:
- `irrelevant_signal`: **29** rijen
- `low_impact_no_signal`: **203** rijen
- `other_no_signal`: **697** rijen

Deze strikte filtering is inhoudelijk gerechtvaardigd: administratieve ingrepen (bv. verhuis, signalisatie, interne stadsdiensten) introduceren ruis voor de centrale onderzoeksvraag rond event-gedreven parkeerdynamiek.

### 12.4 Harmonisatie van types en semantische consistentie
Een belangrijke herwerking in deze versie is de typering van UGent-records:

- `raw_type` wordt eerst genormaliseerd (`raw_type_norm`) met expliciete mapping.
- `event_type` vertrekt primair van die genormaliseerde `raw_type`.
- Gerichte overrides worden enkel toegepast bij semantisch sterke signalen (bv. voetbal/KVM, carnaval, processie).

Daarmee werd een structurele inconsistentie uit eerdere versies beperkt.

Belangrijke inhoudelijke toevoeging:
- nieuw type **`markt`** is nu expliciet aanwezig in `event_type` (niet langer impliciet onder `other` of onbedoeld herlabeld).

In de huidige output bevat `source='ugent_city_dump'` de volgende `event_type`-verdeling:
- `sport`: **285**
- `kermis`: **282**
- `markt`: **80**
- `other`: **63**
- `festival`: **39**
- `football`: **9**
- `procession`: **9**
- `carnival`: **3**

### 12.5 Deduplicatie en integratielogica
Deduplicatie gebeurt op eventniveau met combinatie van:
- datumkoppeling,
- familielabels,
- stringgelijkenis,
- token-overlap,
- en exacte/substring-matches.

Doel: dubbele representatie van reeds gecureerde kern-events vermijden, zonder aanvullende UGent-informatie te verliezen.

In de huidige run:
- gestandaardiseerde UGent-set voor deduplicatie: **805** rijen
- expliciet gedropte UGent-duplicaten t.o.v. baseline: **12** rijen (`events_registry_ugent_dropped_duplicates.parquet`)
- finale geïntegreerde UGent-bijdrage in registry: **770** rijen

Het verschil tussen 805 en 770 komt door combinatie van baseline-deduplicatie en defensieve exacte deduplicatie op dezelfde event-sleutel binnen de geïntegreerde registry.

### 12.6 Finale datasets en inhoud
#### A. `events_registry_final.parquet` (event-level auditlaag)
- Vorm: **1143 rijen x 16 kolommen**
- Bronnen:
  - `ugent_city_dump`: **770**
  - `baseline_04_curated`: **373**

Kolommen:
- `date`
- `event_name`
- `event_type`
- `event_scale`
- `expected_attendance`
- `football_kickoff_hour`
- `data_confidence`
- `is_multiday_event`
- `source`
- `source_priority`
- `external_id`
- `raw_type`
- `raw_status`
- `raw_owner`
- `raw_reference`
- `raw_description`

Deze laag is bedoeld voor methodologische transparantie en eventuele latere reclassificatie/herauditing.

#### B. `events_master.parquet` (dagelijkse modelinput voor MAD)
- Vorm: **2922 rijen x 12 kolommen**
- Datumbereik: **2019-01-01 t/m 2026-12-31**
- Eventdagen (`is_event_day=1`): **693**

Type-indicatoren (som over alle dagen):
- `is_football_day`
- `is_sport_day`
- `is_festival_day`
- `is_procession_day`
- `is_kermis_day`
- `is_markt_day`
- `is_carnival_day`
- `is_other_day`

Kolommen in `events_master`:
- `date`
- `event_scale_max`
- `n_concurrent_events`
- `football_kickoff_hour`
- `data_confidence`
- `event_names_combined`
- `is_football_day`
- `is_sport_day`
- `is_festival_day`
- `is_procession_day`
- `is_kermis_day`
- `is_markt_day`
- `is_carnival_day`
- `is_other_day`
- `is_event_day`

Deze structuur is volledig compatibel met de MAD-assembly in latere fase.

### 12.7 Waarom deze kolommen methodologisch gerechtvaardigd zijn
De gekozen features combineren:
- **intensiteit** (`event_scale_max`),
- **complexiteit** (`n_concurrent_events`),
- **type-effecten** (binaire typeflags),
- **timing-signaal** (`football_kickoff_hour`),
- **datakwaliteitscontext** (`data_confidence`),
- **auditspoor** (`event_names_combined` en event-level registry).

Dit ondersteunt zowel:
- verklarende analyse (welke eventtypen correleren met bezettingsschommelingen),
- als voorspellende modelbouw (feature engineering voor MAD).

### 12.8 Validatie en kwaliteitsborging
De notebook valideert expliciet:
- schema en kolomset,
- unieke datums in `events_master`,
- afwezigheid van kern-NaN in kritieke kolommen,
- binaire integriteit van flag-kolommen,
- continu datumbereik over de volledige projecthorizon.

Daarmee is de output niet alleen inhoudelijk, maar ook technisch geschikt voor downstream gebruik.

### 12.9 Beperkingen en risico's
Ondanks de consolidatie blijven drie beperkingen relevant:
- Typeharmonisatie blijft deels regelgebaseerd en dus gevoelig aan tekstvariatie.
- `expected_attendance` is voor sommige categorieen een heuristische proxy.
- Zonder expliciete georuimtelijke koppeling aan parkingzones blijft locatie-impact op micro-niveau beperkt modelleerbaar.

Deze beperkingen zijn aanvaardbaar in de huidige fase, op voorwaarde dat ze in interpretatie en modeldiscussie transparant worden gerapporteerd.

### 12.10 Gebruik in volgende fases (MAD en modellering)
`events_master.parquet` is de operationele eventfeature-tabel voor de volgende pipelinefase.
In MAD-assembly wordt deze tabel op datum gemerged zodat elk uurrecord verrijkt wordt met dagelijkse eventcontext.

Praktisch betekent dit dat latere analyses en modellen kunnen testen:
- effect van eventtypes,
- effect van eventintensiteit,
- effect van gelijktijdige events,
- en robuustheid over confidence-niveaus.

Hiermee vormt deze notebook een volledige, reproduceerbare en thesis-conforme eventdatabasis voor de rest van het onderzoekstraject.


